# RBM Data Preparation

Prepare MovieLens ratings for RBM training: chronological train/test split and user–movie matrix.

**Note**: The full MovieLens-20M ratings can be too large for a pivot table on many laptops. Use the limits below to avoid kernel crashes, then increase gradually if needed.

## Step 1 — Train/Test Split (chronological)

Requirements:
- Use `rating.csv` (fallback: `ratings.csv`).
- Convert `timestamp` to datetime.
- Sort by time and split 80% train / 20% test.
- Ensure test only contains users and movies seen in training.


In [6]:
import os
from pathlib import Path
import pandas as pd

# Resolve project root regardless of where the notebook runs
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data"

ratings_path = data_dir / "rating.csv"
if not ratings_path.exists():
    ratings_path = data_dir / "ratings.csv"

if not ratings_path.exists():
    raise FileNotFoundError("rating.csv (or ratings.csv) not found")

# Step 1 — Chronological Train/Test Split
ratings = pd.read_csv(ratings_path)
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], errors="coerce")
print("timestamp NaT count:", ratings["timestamp"].isna().sum())
ratings = ratings.dropna(subset=["timestamp"]).copy()
ratings = ratings.sort_values("timestamp").reset_index(drop=True)

split_idx = int(len(ratings) * 0.8)
train_ratings = ratings.iloc[:split_idx].copy()
test_ratings = ratings.iloc[split_idx:].copy()

# Step 2 — Filter Test Users Only (no movie filtering)
train_users = set(train_ratings["userId"].unique())
test_ratings = test_ratings[
    test_ratings["userId"].isin(train_users)
].copy()

# Step 3 — Print Dataset Statistics
print("train_ratings.shape:", train_ratings.shape)
print("test_ratings.shape:", test_ratings.shape)

print("train users:", train_ratings["userId"].nunique())
print("train movies:", train_ratings["movieId"].nunique())

print("test users:", test_ratings["userId"].nunique())
print("test movies:", test_ratings["movieId"].nunique())

# Step 4 — Print the Chronological Split Time
split_time = ratings.iloc[split_idx]["timestamp"]
print("================================")
print("Train/Test split timestamp:", split_time)
print("train max time:", train_ratings["timestamp"].max())
print("test min time:", test_ratings["timestamp"].min())
print("================================")

# Step 5 — Save the Results
output_dir = data_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
train_ratings.to_csv(output_dir / "train_ratings.csv", index=False)
test_ratings.to_csv(output_dir / "test_ratings.csv", index=False)
print("Saved splits to:", output_dir)


timestamp NaT count: 0
train_ratings.shape: (16000210, 4)
test_ratings.shape: (739053, 4)
train users: 112466
train movies: 12388
test users: 5643
test movies: 23012
Train/Test split timestamp: 2009-10-20 07:33:43
train max time: 2009-10-20 07:33:38
test min time: 2009-10-20 07:33:43
Saved splits to: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed


## Step 2 — Build and Save User × Movie Matrix

Use the training subset to build the user–movie matrix with `pivot_table`, inspect its shape, preview a few rows, and save it to `data/processed/`.

In [7]:
# Limit dataset size before pivoting
MAX_USERS = 2000
MAX_MOVIES = 4000

# Select first MAX_USERS users from training
first_users = train_ratings["userId"].drop_duplicates().head(MAX_USERS)
train_subset = train_ratings[train_ratings["userId"].isin(first_users)].copy()

# From that subset, select top MAX_MOVIES movies by rating count
top_movies = train_subset["movieId"].value_counts().head(MAX_MOVIES).index
train_subset = train_subset[train_subset["movieId"].isin(top_movies)].copy()

print("about to pivot training subset...")
print("train_subset.shape:", train_subset.shape)
print("subset users:", train_subset["userId"].nunique())
print("subset movies:", train_subset["movieId"].nunique())

# Build user–movie matrix from subset
user_movie_matrix = train_subset.pivot_table(
    index="userId",
    columns="movieId",
    values="rating",
    aggfunc="mean"
)

print("user_movie_matrix.shape:", user_movie_matrix.shape)
display(user_movie_matrix.head())

# Save results
user_movie_matrix.to_csv(output_dir / "user_movie_matrix_train.csv")
print("Saved matrix to:", output_dir / "user_movie_matrix_train.csv")


about to pivot training subset...
train_subset.shape: (112193, 4)
subset users: 2000
subset movies: 1320
user_movie_matrix.shape: (2000, 1320)


movieId,1,2,3,4,5,6,7,8,9,10,...,2443,2593,3265,3266,4006,4424,4970,5060,6425,6918
userId,,,,,,,,,,,,,,,,,,,,,
68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
172,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
452,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved matrix to: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/user_movie_matrix_train.csv
